# KnobNet - Case C: Joint Full Fine-tuning

**실험 목적**: input + reference audio를 concat해 MERT가 두 오디오를 **동시에** 바라보도록 학습.  
Dual Full FT(Case B)와 비교해 joint 인코딩이 knob 예측에 유리한지 측정.

**논문 흐름**: MERT 없음 → MERT frozen HEAD-only → MERT full fine-tune (Case B) → **Joint full fine-tune (이 노트북)**

**Case B와의 차이**:
- `KnobNet` (Dual: input/ref 각각 MERT 2회) → `KnobNetJoint` (input+ref concat → MERT 1회)
- 나머지 하이퍼파라미터, lr 전략, early stop 모두 동일

**lr 설정**: HEAD 1e-3 / MERT 1e-5 (mert_lr_scale=0.01, Discriminative Fine-tuning)  
**주의**: 1 epoch ≈ 30분. early stop(patience=10)으로 수렴 시 자동 종료됩니다.

**순서**: 환경 설치 → Drive 마운트 → 데이터 해제 → 학습 → export

## 1. 환경 설치

In [ ]:
!pip install -q transformers soundfile torchaudio

## 2. GitHub 클론

In [ ]:
import os

GITHUB_REPO = "https://github.com/kuhberaBubo/KnobNet.git"  # <-- 수정
PROJECT_DIR = "/content/KnobNet"

if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("현재 디렉터리:", os.getcwd())

## 3. Google Drive 마운트 & 데이터 압축 해제

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import re
from pathlib import Path

ZIP_FILES = [
    "/content/drive/MyDrive/Colab/dataset/26.05.21.zip",  # <-- 수정
]

DATA_DIR = Path(PROJECT_DIR) / "data"
DATA_DIR.mkdir(exist_ok=True)

!apt-get install -q p7zip-full

for zip_file in ZIP_FILES:
    zip_path = Path(zip_file)
    if re.fullmatch(r'\.z\d+', zip_path.suffix, re.IGNORECASE):
        print(f"[skip] 연속 파트: {zip_path.name}")
        continue
    if not zip_path.exists():
        print(f"[SKIP] 파일 없음: {zip_path}")
        continue
    print(f"압축 해제 중: {zip_path.name} ...")
    !7z x "{zip_file}" -o"{DATA_DIR}" -y
    print("완료")

print("\n데이터 디렉터리 구조:")
!find {DATA_DIR} -maxdepth 3 -type d

## 4. 데이터 확인

In [ ]:
from pathlib import Path

DATA_DIR = Path(PROJECT_DIR) / "data"

print("=== data/ 하위 전체 디렉터리 ===")
!find {DATA_DIR} -type d | sort

print("\n=== samples.csv 위치 ===")
!find {DATA_DIR} -name "samples.csv" | sort

print("\n=== wav 파일 수 ===")
!find {DATA_DIR} -name "*.wav" | wc -l

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from dataset.loader import make_loaders

WET_DIR = "data/26.05.21/output/black"  # <-- 수정

# Joint full fine-tuning도 MERT unfrozen → 캐시 불가, 원본 오디오 로더 사용
train_loader, val_loader = make_loaders(
    dataset_root = PROJECT_DIR,
    wet_dir      = WET_DIR,
    batch_size   = 16,
    val_split    = 0.2,
    num_workers  = 2,
)

from train.train import print_batch
inp, ref, knobs = next(iter(train_loader))
print_batch(inp, ref, knobs)

## 5. 모델 초기화

`KnobNetJoint`: input + reference를 concat (B, 2T) → MERT 1회 통과 → 시간축 절반씩 분리해 I_feat / O_feat 추출  
HEAD 구조는 `KnobNet`(Case B)과 완전히 동일하며, MERT 인코딩 방식만 다름.

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

from model.model import KnobNetJoint
from utils.config import KNOB_PARAMS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# freeze_mert=False: 처음부터 MERT 전체 학습
# layer_idx=4: layer 실험에서 선택된 최적 레이어
model = KnobNetJoint(num_knobs=len(KNOB_PARAMS), freeze_mert=False, layer_idx=4).to(device)
model.summary()

## 6. 학습 (Joint MERT + HEAD end-to-end)

- **MAX_EPOCHS**: 80 (early stop으로 실질 시간 제어)
- **PATIENCE**: 10
- **HEAD lr**: 1e-3 / **MERT lr**: 1e-5 (Case B와 동일한 lr 전략)
- **scheduler**: CosineAnnealingLR (두 lr 모두 epoch에 따라 ~0으로 감소)

In [ ]:
from train.train import (
    run_epoch, evaluate_all,
    make_optimizer, save_checkpoint, load_checkpoint,
    log_epoch, log_param_mae, log_accuracy, log_csv,
)

CKPT_DIR  = Path("/content/drive/MyDrive/KnobNet/ablation/joint_full_ft")  # <-- 수정
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / "joint_full_ft_best.pt"
CSV_PATH  = CKPT_DIR / "joint_full_ft_log.csv"

MAX_EPOCHS = 80
PATIENCE   = 10

criterion = nn.L1Loss()

# HEAD lr=1e-3, MERT lr=1e-5 (100배 차이, Case B와 동일)
# 근거: Discriminative Fine-tuning (ULMFiT, 2018)
optimizer = make_optimizer(model, lr=1e-3, phase=2, mert_lr_scale=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler    = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

best_val    = float("inf")
no_improve  = 0
start_epoch = 1

if CKPT_PATH.exists():
    meta        = load_checkpoint(CKPT_PATH, model, optimizer, scheduler)
    start_epoch = meta["epoch"] + 1
    best_val    = meta["val_loss"]
    print(f"체크포인트 재개: epoch={meta['epoch']}  best_val={best_val:.4f}")

print(f"학습: epoch {start_epoch} ~ {MAX_EPOCHS}  patience={PATIENCE}  AMP={'on' if scaler else 'off'}")
model.summary()

In [ ]:
if start_epoch == 1 and CSV_PATH.exists():
    CSV_PATH.unlink()
    print("기존 CSV 초기화 (새 학습 시작)")

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, device, scaler)
    metrics    = evaluate_all(model, val_loader, criterion, device, tolerance=0.1)
    val_loss   = metrics["val_loss"]
    scheduler.step()

    improved = val_loss < best_val
    if improved:
        best_val   = val_loss
        no_improve = 0
        save_checkpoint(CKPT_PATH, model, epoch, phase=2, val_loss=val_loss,
                        optimizer=optimizer, scheduler=scheduler)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch} (patience={PATIENCE})")
            break

    log_epoch(2, epoch, MAX_EPOCHS, train_loss, val_loss, improved)
    log_param_mae(metrics["mae"])
    log_accuracy(metrics["acc"], tolerance=0.1)
    log_csv(CSV_PATH, 2, epoch, train_loss, val_loss, metrics["mae"], metrics["acc"])

print("\n학습 완료")
print(f"best val loss: {best_val:.4f}")

## 7. 최종 모델 export

In [ ]:
EXPORT_PATH = CKPT_DIR / "joint_full_ft_final.pt"

load_checkpoint(CKPT_PATH, model)
model.export(EXPORT_PATH)

size_mb = EXPORT_PATH.stat().st_size / 1024 / 1024
print(f"저장 완료: {EXPORT_PATH}  ({size_mb:.1f} MB)")

## (선택) 예측 결과 확인

In [ ]:
from train.train import print_predictions

load_checkpoint(CKPT_PATH, model)
print_predictions(model, val_loader, device, n=8)